In [1]:
# rm(list = ls())
# gc()
library(MASS) # 多元正态分布
library(Rcpp) # R链接c++语言
# library(ggplot2) # 绘图
# library(reshape2) # 数据融合
library(compiler)
library(survival)
# 平行计算
library(foreach)
library(iterators)
library(parallel)
library(doParallel) # 仅此一个也行
sourceCpp('TestVCM.cpp')
# sourceCpp('TestCPLRM3.cpp')
# 矩阵计算函数(未除以n): angle_matrix()
# 带Bootstrap的统计量仿真函数: testCPLRM()
r_files <- list.files('speedyCMI/')
for (file in r_files) {
    source(paste0('speedyCMI/', file))    
}

R.version

               _                           
platform       x86_64-pc-linux-gnu         
arch           x86_64                      
os             linux-gnu                   
system         x86_64, linux-gnu           
status                                     
major          3                           
minor          5.2                         
year           2018                        
month          12                          
day            20                          
svn rev        75870                       
language       R                           
version.string R version 3.5.2 (2018-12-20)
nickname       Eggshell Igloo              

In [2]:
data = read.csv('ACTG_315/HIV317.csv')
data$delta = 1 - is.na(data$cd4)
data[is.na(data$cd4), ]$cd4 = mean(data[!is.na(data$cd4), ]$cd4)
data <- data[data$day<=90,]
data <- data[-c(9),] # 异常值

In [3]:
Y = data$lgcopy
X = cbind(rep(1, dim(data)[1]))
T1 = data$cd4
U <- data$day/30
# 标准化
# Y <- scale(Y)
# X <- apply(X, 2, FUN = function(x) {scale(x)})
# T1 <- scale(T1)
# U <- scale(U)

delta = data$delta
dat = data.frame(z = X, w = T1, y = Y, d = delta)
imp_dat1 = cmi_fp_analytical(imputation_model = Surv(time = w, event = d) ~ z,
                          dist = "exponential",
                          data = dat)
imp_dat2 = cmi_fp_analytical(imputation_model = Surv(time = w, event = d) ~ z,
                          dist = "weibull",
                          data = dat)

print(c('censoring rate:', round(1 - mean(delta), 4)))
print(c('sample size:', length(Y)))

[1] "censoring rate:" "0.2032"         
[1] "sample size:" "315"         


In [4]:
method <- "Test_VCM"
h_con <- 1.54 # 带宽系数
# n_con <- 3
m <- 50
B <- 2000
df1 <- list(Y = Y, X = X, T1 = T1, delta = delta, U = U, V = T1,
                   VE = imp_dat1$imputed_data$imp, VW = imp_dat2$imputed_data$imp,
                   B = B, h_con = h_con, m = m)

print(c('censoring rate:', round(1 - mean(df1$delta), 4)))
print(c('sample size:', length(df1$Y)))

res <- do.call(method, df1)
cat("带宽:", h_con, '\t', "p-值:", '\n')
res$p_value

[1] "censoring rate:" "0.2032"         
[1] "sample size:" "315"         
带宽: 1.54 	 p-值: 


0.691,0.672,0.5885,0.7,0.2205,0.5255,0.704,0.2305,0.6305


In [5]:
method <- "Test_VCM"
h_con <- 3.14 # 带宽系数
# n_con <- 3
m <- 50
B <- 2000
df1 <- list(Y = Y, X = X, T1 = T1, delta = delta, U = U, V = T1,
                   VE = imp_dat1$imputed_data$imp, VW = imp_dat2$imputed_data$imp,
                   B = B, h_con = h_con, m = m)

print(c('censoring rate:', round(1 - mean(df1$delta), 4)))
print(c('sample size:', length(df1$Y)))

res <- do.call(method, df1)
cat("带宽:", h_con, '\t', "p-值:", '\n')
res$p_value

[1] "censoring rate:" "0.2032"         
[1] "sample size:" "315"         
带宽: 3.14 	 p-值: 


0.5055,0.65,0.607,0.5035,0.274,0.493,0.5035,0.338,0.642


In [6]:
method <- "Test_VCM"
h_con <- 2.34 # 带宽系数
# n_con <- 3
m <- 50
B <- 2000
df1 <- list(Y = Y, X = X, T1 = T1, delta = delta, U = U, V = T1,
                   VE = imp_dat1$imputed_data$imp, VW = imp_dat2$imputed_data$imp,
                   B = B, h_con = h_con, m = m)

print(c('censoring rate:', round(1 - mean(df1$delta), 4)))
print(c('sample size:', length(df1$Y)))

res <- do.call(method, df1)
cat("带宽:", h_con, '\t', "p-值:", '\n')
res$p_value

[1] "censoring rate:" "0.2032"         
[1] "sample size:" "315"         
带宽: 2.34 	 p-值: 


0.875,0.7785,0.6965,0.8765,0.538,0.7765,0.8755,0.5305,0.7535


In [7]:
h_con <- 2.34
df1 <- list(Y = Y, X = X, T1 = T1, delta = delta, U = U, V = T1,
                   VE = imp_dat1$imputed_data$imp, VW = imp_dat2$imputed_data$imp,
                   B = 1, h_con = h_con, m = m)
res111 <- do.call(method, df1)

In [8]:
n2 <- length(Y)
p2 <- dim(X)[2]
m <- 6

yhats2 <- t(matrix(rep(1, n2, each = m), m))
errors2 <- t(matrix(rep(1, n2, each = m), m))


yhats2[, 1] <- res111$yhat1
errors2[, 1] <- res111$resid1w

yhats2[, 2] <- c(res111$yhat2, rep(0, n2 - length(res111$yhat2)))
errors2[, 2] <- c(res111$resid2, rep(0, n2 - length(res111$resid2)))

yhats2[, 3] <- res111$yhat3
errors2[, 3] <- res111$resid3

yhats2[, 4] <- res111$yhat4
errors2[, 4] <- res111$resid4

yhats2[, 5] <- res111$yhat5
errors2[, 5] <- res111$resid5

yhats2[, 6] <- res111$yhat6
errors2[, 6] <- res111$resid6



write.table(yhats2, file = 'results/yhats1.txt', row.names = FALSE, col.names = FALSE)
write.table(errors2, file = 'results/errors1.txt', row.names = FALSE, col.names = FALSE)
